# Comparison of Segmentation Methods on Oxford-IIIT Pet

This notebook evaluates and compares three segmentation approaches on the **Oxford-IIIT Pet** dataset (trimap: pet, background, boundary):

1. **Pre-trained U-Net** – Encoder–decoder with ResNet backbone.  
2. **Vision Transformer (ViT)** from *timm* – Transformer adapted for segmentation.  
3. **Cross-Teaching Ensemble** – U-Net and ViT exchange pseudo-labels; combined prediction.

We benchmark on the same data, use Dice and IoU, and visualize predictions. Data is loaded via `data_oxford_pet` (TFDS).


Imports

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Same data split as training scripts (DATA_SEED)
from data_oxford_pet import get_train_val_test_loaders, get_fixed_splits, NUM_SEGMENTATION_CLASSES
from comparison_utils import (
    load_unet, load_vit, EnsembleInference,
    predict_unet, predict_vit,
    dice_score_macro, iou_score_macro, evaluate_models,
    show_image, show_image_and_mask, compare_predictions,
    get_checkpoints_dir,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

Load and Visualize Data

In [11]:
# Oxford-IIIT Pet: use shared data module. Resize 512 for comparison.
RESIZE_EVAL = 512
_, _, test_loader = get_train_val_test_loaders(target_size=RESIZE_EVAL, batch_size=1, val_fraction=0.1)
# wrap so we can index for viz (take first batch = one sample)
test_ds = test_loader.dataset

In [13]:
dataset = test_ds
sample_img, sample_mask = dataset[0]
sample_img = sample_img.unsqueeze(0).to(DEVICE)

AssertionError: No images found!

In [ ]:
img, mask = dataset[0]
show_image_and_mask(img, mask)

Load in Models and Model Architectures

In [ ]:
# Models and metrics are in comparison_utils (and CrossTeachingTraining). Load checkpoints below.

In [ ]:
# Load from project checkpoints/ (created by Unet_TransferLearn.py, ViT_train.py, CrossTeachingTraining.py)
ckpt = get_checkpoints_dir()
if not ckpt.exists():
    print("Run Unet_TransferLearn.py and ViT_train.py (or CrossTeachingTraining.py) first to create checkpoints.")
unet = load_unet(device=DEVICE)
vit = load_vit(device=DEVICE)
ensemble = EnsembleInference(unet, vit, device=DEVICE)

Evaluation and best-case comparison

In [ ]:
# Full test-set evaluation (same DATA_SEED split as training)
metrics, summary = evaluate_models(test_ds, unet, vit, ensemble, device=DEVICE)
table = pd.DataFrame([{"Model": k, "Dice": v[0], "IoU": v[1]} for k, v in summary.items()])
display(table)
best = max(summary.items(), key=lambda x: x[1][0])
print(f"\n*** Best case: {best[0]} (Dice={best[1][0]:.4f}, IoU={best[1][1]:.4f}) ***")

Visualize

In [ ]:
idx = 0
image, mask = dataset[idx]

image_tensor = image.unsqueeze(0).to(DEVICE)   # shape: (1, 1, H, W)
mask_tensor  = mask.unsqueeze(0).to(DEVICE)

# Get model predictions
unet_pred = predict_unet(unet, image_tensor, device=DEVICE)
vit_pred = predict_vit(vit, image_tensor, device=DEVICE)
ensemble_pred = ensemble.predict(image_tensor)

# Visualize
compare_predictions(image_tensor, mask_tensor, unet_pred, vit_pred, ensemble_pred)

In [ ]:
pred_unet = predict_unet(unet, sample_img)
pred_vit = predict_vit(vit, sample_img)
pred_ensemble = ensemble.predict(sample_img)

Evaluations

In [ ]:
# Metrics and evaluate_models from comparison_utils (imported above).

In [ ]:
# Optional: run evaluation again. Main results and best-case are in the cell above.